In [2]:
import rasterio

raster_path = r"D:/Arq-Azzoni/UrbanSprawl/Bases_dados/Landsat/landsat_sp_annual/landsat_annual_sp_2000_mosaic.tif"

with rasterio.open(raster_path) as src:
    print("Número de bandas:", src.count)
    print("Descriptions:", src.descriptions)
    for i in range(1, src.count + 1):
        print(i, src.descriptions[i - 1], src.tags(i))

Número de bandas: 7
Descriptions: ('ndvi_max_annual', 'ndvi_median_annual', 'ndbi_median_annual', 'tempC_mean_clear_annual', 'tempC_hot_season_median_annual', 'n_obs_annual', 'mask_valid_annual')
1 ndvi_max_annual {}
2 ndvi_median_annual {}
3 ndbi_median_annual {}
4 tempC_mean_clear_annual {}
5 tempC_hot_season_median_annual {}
6 n_obs_annual {}
7 mask_valid_annual {}


In [ ]:
import rasterio
import pandas as pd
import numpy as np

raster_path = r"D:/Arq-Azzoni/UrbanSprawl/Bases_dados/Landsat/landsat_sp_annual/landsat_annual_sp_2000_mosaic.tif"

with rasterio.open(raster_path) as src:
    print("===== INFORMAÇÕES GERAIS =====")
    print("Arquivo:", raster_path)
    print("Driver:", src.driver)
    print("CRS:", src.crs)
    print("Largura (cols):", src.width)
    print("Altura (rows):", src.height)
    print("Número de bandas:", src.count)
    print("Dtypes:", src.dtypes)
    print("Nodata:", src.nodata)
    print("Bounds:", src.bounds)
    print("Resolução:", src.res)
    print("Transform:", src.transform)

    print("\n===== DESCRIÇÃO DAS BANDAS =====")
    for i in range(1, src.count + 1):
        desc = src.descriptions[i - 1]
        tags = src.tags(i)
        print(f"Banda {i}:")
        print("  description:", desc)
        print("  tags:", tags)

    print("\n===== TAGS GERAIS =====")
    print(src.tags())

    print("\n===== RESUMO RÁPIDO DAS BANDAS =====")
    resumo = []
    for i in range(1, src.count + 1):
        arr = src.read(i).astype("float32")

        if src.nodata is not None:
            arr[arr == src.nodata] = np.nan

        resumo.append({
            "banda": i,
            "description": src.descriptions[i - 1],
            "dtype": src.dtypes[i - 1],
            "min": np.nanmin(arr),
            "max": np.nanmax(arr),
            "mean": np.nanmean(arr),
            "std": np.nanstd(arr),
            "n_nan": np.isnan(arr).sum()
        })

    resumo_df = pd.DataFrame(resumo)
    print(resumo_df)

In [ ]:
import rasterio
import pandas as pd
import numpy as np

raster_path = r"D:/Arq-Azzoni/UrbanSprawl/Bases_dados/Landsat/landsat_sp_annual/landsat_annual_sp_2000_mosaic.tif"

with rasterio.open(raster_path) as src:
    resumo = []
    
    for i in range(1, src.count + 1):
        arr = src.read(i).astype("float32")
        arr[arr == src.nodata] = np.nan
        
        resumo.append({
            "banda": i,
            "min": float(np.nanmin(arr)),
            "p1": float(np.nanpercentile(arr, 1)),
            "p5": float(np.nanpercentile(arr, 5)),
            "p50": float(np.nanpercentile(arr, 50)),
            "p95": float(np.nanpercentile(arr, 95)),
            "p99": float(np.nanpercentile(arr, 99)),
            "max": float(np.nanmax(arr)),
            "mean": float(np.nanmean(arr)),
            "std": float(np.nanstd(arr))
        })

resumo_df = pd.DataFrame(resumo)
print(resumo_df)

   banda        min         p1         p5        p50        p95        p99  \
0      1  -0.424729   0.104634   0.497484   0.749742   0.897551   0.920171   
1      2  -0.575188  -0.153112   0.234826   0.521842   0.824407   0.854014   
2      3  -0.928433  -0.492781  -0.401834  -0.082996   0.102714   0.156340   
3      4  -4.105720  19.008722  21.598930  28.939062  33.881641  35.895531   
4      5 -52.467285  23.633223  26.246298  33.753979  40.191818  44.112289   
5      6   1.000000   7.000000  11.000000  22.000000  48.000000  64.000000   
6      7   1.000000   1.000000   1.000000   1.000000   1.000000   1.000000   

         max       mean        std  
0   0.999960   0.728882   0.147955  
1   0.999108   0.530776   0.192553  
2   0.582965  -0.119707   0.166143  
3  67.221863  28.471798   3.770870  
4  87.934723  33.482841   4.316836  
5  81.000000  25.768101  11.915395  
6   1.000000   1.000000   0.000000  


In [ ]:
import re
from pathlib import Path
# ---------------------------------
# Folder with all TIFFs
# ---------------------------------
folder = Path(r"D:\Users\ivan.cavalcanti\Documents\Projects\extract_landsat\outputs\drive")

# ---------------------------------
# Find matching files
# ---------------------------------
pattern = re.compile(r"jundiai_(\d{4})Q([1-4])_ndvi_temp\.tif$", re.IGNORECASE)

files = []
for path in folder.glob("*.tif"):
    m = pattern.search(path.name)
    if m:
        year = int(m.group(1))
        quarter = int(m.group(2))
        files.append((year, quarter, path))

files = sorted(files)

print("Found files:")
for year, quarter, path in files:
    print(year, quarter, path.name)

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt


# ---------------------------------
# Helpers
# ---------------------------------
def read_band_by_description(src, band_name):
    desc = list(src.descriptions)
    if band_name not in desc:
        raise ValueError(f"Band '{band_name}' not found in {src.name}. Found: {desc}")
    idx = desc.index(band_name) + 1
    return src.read(idx)

def summarize_raster(path, year, quarter):
    with rasterio.open(path) as src:
        ndvi = read_band_by_description(src, "ndvi")
        tempC = read_band_by_description(src, "tempC")
        n_obs = read_band_by_description(src, "n_obs")

        total_pixels = src.width * src.height

        valid_ndvi = ~np.isnan(ndvi)
        valid_temp = ~np.isnan(tempC)
        valid_nobs = ~np.isnan(n_obs)

        row = {
            "year": year,
            "quarter": quarter,
            "file": path.name,
            "width": src.width,
            "height": src.height,
            "total_pixels": total_pixels,

            "valid_ndvi_pixels": int(valid_ndvi.sum()),
            "valid_temp_pixels": int(valid_temp.sum()),
            "valid_nobs_pixels": int(valid_nobs.sum()),

            "valid_ndvi_pct": 100 * valid_ndvi.sum() / total_pixels,
            "valid_temp_pct": 100 * valid_temp.sum() / total_pixels,
            "valid_nobs_pct": 100 * valid_nobs.sum() / total_pixels,

            "mean_ndvi": np.nanmean(ndvi),
            "median_ndvi": np.nanmedian(ndvi),
            "min_ndvi": np.nanmin(ndvi),
            "max_ndvi": np.nanmax(ndvi),

            "mean_tempC": np.nanmean(tempC),
            "median_tempC": np.nanmedian(tempC),
            "min_tempC": np.nanmin(tempC),
            "max_tempC": np.nanmax(tempC),

            "mean_n_obs": np.nanmean(n_obs),
            "median_n_obs": np.nanmedian(n_obs),
            "min_n_obs": np.nanmin(n_obs),
            "max_n_obs": np.nanmax(n_obs),
        }
        return row

# ---------------------------------
# Build DataFrame
# ---------------------------------
rows = [summarize_raster(path, year, quarter) for year, quarter, path in files]
df = pd.DataFrame(rows).sort_values(["year", "quarter"]).reset_index(drop=True)

print("\nSummary table:")
print(df)

# Save summary to CSV
csv_path = folder / "landsat_summary_2000_2010_2022.csv"
df.to_csv(csv_path, index=False)
print(f"\nSaved summary CSV to: {csv_path}")

In [ ]:
def plot_pivot_heatmap(dataframe, value_col, title, fmt="{:.2f}"):
    pivot = dataframe.pivot(index="year", columns="quarter", values=value_col).sort_index()
    arr = pivot.values

    plt.figure(figsize=(6, 3.5))
    im = plt.imshow(arr, aspect="auto")
    plt.colorbar(im, label=value_col)
    plt.xticks(range(len(pivot.columns)), pivot.columns)
    plt.yticks(range(len(pivot.index)), pivot.index)
    plt.xlabel("Quarter")
    plt.ylabel("Year")
    plt.title(title)

    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            plt.text(j, i, fmt.format(arr[i, j]), ha="center", va="center")

    plt.show()

plot_pivot_heatmap(df, "mean_ndvi", "Mean NDVI")
plot_pivot_heatmap(df, "mean_tempC", "Mean temperature (°C)")
plot_pivot_heatmap(df, "mean_n_obs", "Mean n_obs")
plot_pivot_heatmap(df, "valid_temp_pct", "Valid temperature coverage (%)")

In [ ]:
# ---------------------------------
# Line plots by quarter for each year
# ---------------------------------
years = sorted(df["year"].unique())

plt.figure(figsize=(8, 4))
for year in years:
    sub = df[df["year"] == year].sort_values("quarter")
    plt.plot(sub["quarter"], sub["mean_ndvi"], marker="o", label=str(year))
plt.xticks([1, 2, 3, 4])
plt.xlabel("Quarter")
plt.ylabel("Mean NDVI")
plt.title("Mean NDVI by quarter")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
for year in years:
    sub = df[df["year"] == year].sort_values("quarter")
    plt.plot(sub["quarter"], sub["mean_tempC"], marker="o", label=str(year))
plt.xticks([1, 2, 3, 4])
plt.xlabel("Quarter")
plt.ylabel("Mean temperature (°C)")
plt.title("Mean temperature by quarter")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
for year in years:
    sub = df[df["year"] == year].sort_values("quarter")
    plt.plot(sub["quarter"], sub["mean_n_obs"], marker="o", label=str(year))
plt.xticks([1, 2, 3, 4])
plt.xlabel("Quarter")
plt.ylabel("Mean n_obs")
plt.title("Mean valid observations by quarter")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
for year in years:
    sub = df[df["year"] == year].sort_values("quarter")
    plt.plot(sub["quarter"], sub["valid_temp_pct"], marker="o", label=str(year))
plt.xticks([1, 2, 3, 4])
plt.xlabel("Quarter")
plt.ylabel("Valid temperature pixels (%)")
plt.title("Valid coverage by quarter")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# Change from Q1 to Q4 within each year
q1 = df[df["quarter"] == 1].set_index("year")
q4 = df[df["quarter"] == 4].set_index("year")

seasonal_change = pd.DataFrame({
    "ndvi_q4_minus_q1": q4["mean_ndvi"] - q1["mean_ndvi"],
    "temp_q4_minus_q1": q4["mean_tempC"] - q1["mean_tempC"],
    "nobs_q4_minus_q1": q4["mean_n_obs"] - q1["mean_n_obs"],
    "valid_temp_pct_q4_minus_q1": q4["valid_temp_pct"] - q1["valid_temp_pct"],
})

print(seasonal_change)

In [ ]:
# ---------------------------------
# Build lookup from (year, quarter) -> path
# ---------------------------------
path_lookup = {(year, quarter): path for year, quarter, path in files}

def read_metric(path, metric):
    with rasterio.open(path) as src:
        arr = read_band_by_description(src, metric)
        bounds = src.bounds
        extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]
        return arr, extent

def plot_difference_map(path_a, path_b, metric, title, label):
    arr_a, extent_a = read_metric(path_a, metric)
    arr_b, extent_b = read_metric(path_b, metric)

    if arr_a.shape != arr_b.shape:
        raise ValueError(f"Shape mismatch: {arr_a.shape} vs {arr_b.shape}")

    diff = arr_b - arr_a
    v = np.nanpercentile(np.abs(diff), 98)

    plt.figure(figsize=(7, 7))
    plt.imshow(
        np.ma.masked_invalid(diff),
        extent=extent_a,
        origin="upper",
        cmap="RdBu_r",
        vmin=-v,
        vmax=v
    )
    plt.colorbar(label=label)
    plt.title(title)
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.show()

# Example 1: 2022 Q1 minus 2000 Q1 for temperature
plot_difference_map(
    path_lookup[(2000, 1)],
    path_lookup[(2022, 1)],
    metric="tempC",
    title="Temperature change: 2022 Q1 - 2000 Q1",
    label="Δ temperature (°C)"
)

# Example 2: 2022 Q1 minus 2000 Q1 for NDVI
plot_difference_map(
    path_lookup[(2000, 1)],
    path_lookup[(2022, 1)],
    metric="ndvi",
    title="NDVI change: 2022 Q1 - 2000 Q1",
    label="Δ NDVI"
)

# Example 3: Q4 minus Q1 within 2022
plot_difference_map(
    path_lookup[(2022, 1)],
    path_lookup[(2022, 4)],
    metric="ndvi",
    title="NDVI seasonal change: 2022 Q4 - 2022 Q1",
    label="Δ NDVI"
)

In [ ]:
compact = df[
    [
        "year", "quarter",
        "mean_ndvi", "median_ndvi",
        "mean_tempC", "median_tempC",
        "mean_n_obs", "valid_temp_pct"
    ]
].sort_values(["year", "quarter"])

print(compact)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from pathlib import Path

base = Path(r"D:\Users\ivan.cavalcanti\Documents\Projects\extract_landsat\outputs\drive")

rasters = {
    ("2000", "Q1"): base / "jundiai_2000Q1_ndvi_temp.tif",
    ("2000", "Q2"): base / "jundiai_2000Q2_ndvi_temp.tif",
    ("2000", "Q4"): base / "jundiai_2000Q4_ndvi_temp.tif",
    ("2022", "Q1"): base / "jundiai_2022Q1_ndvi_temp.tif",
    ("2022", "Q2"): base / "jundiai_2022Q2_ndvi_temp.tif",
    ("2022", "Q4"): base / "jundiai_2022Q4_ndvi_temp.tif",
}

municipios_path = Path(r"D:\Arq-Azzoni\UrbanSprawl\Bases_dados\Shapes\BR_Municipios_2024\BR_Municipios_2024.shp")

def read_band_by_description(src, band_name):
    desc = list(src.descriptions)
    if band_name not in desc:
        raise ValueError(f"Banda '{band_name}' não encontrada em {src.name}. Bandas: {desc}")
    idx = desc.index(band_name) + 1
    return idx

def clip_band_to_polygon(raster_path, polygon_gdf, band_name="tempC"):
    with rasterio.open(raster_path) as src:
        band_idx = read_band_by_description(src, band_name)

        poly = polygon_gdf.to_crs(src.crs)
        geoms = [geom.__geo_interface__ for geom in poly.geometry]

        clipped, transform = mask(src, geoms, crop=True, filled=True, indexes=band_idx)

        arr = clipped.astype("float32")
        if arr.ndim == 3:
            arr = arr[0]

        nodata = src.nodata
        if nodata is not None:
            arr[arr == nodata] = np.nan

        h, w = arr.shape
        left = transform.c
        top = transform.f
        right = left + transform.a * w
        bottom = top + transform.e * h
        extent = [left, right, bottom, top]

        return arr, extent, src.crs

def plot_diff_map(ax, diff, extent, title, cmap="RdBu_r", vmax=None):
    arr = np.ma.masked_invalid(diff)

    if vmax is None:
        vmax = np.nanpercentile(np.abs(diff), 98)

    im = ax.imshow(
        arr,
        extent=extent,
        origin="upper",
        cmap=cmap,
        vmin=-vmax,
        vmax=vmax
    )
    ax.set_title(title)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    return im

# carregar e filtrar Jundiaí
jundiai = gpd.read_file(municipios_path)
jundiai = jundiai[jundiai["CD_MUN"].astype(str) == "3525904"].copy()

if jundiai.empty:
    raise ValueError("Não encontrei Jundiaí no arquivo com CD_MUN = 3525904.")

quarters = ["Q1", "Q2", "Q4"]
band_name = "tempC"   # troque para "ndvi" se quiser

results = {}

for q in quarters:
    arr_2000, extent_2000, _ = clip_band_to_polygon(rasters[("2000", q)], jundiai, band_name=band_name)
    arr_2022, extent_2022, _ = clip_band_to_polygon(rasters[("2022", q)], jundiai, band_name=band_name)

    if arr_2000.shape != arr_2022.shape:
        raise ValueError(f"Shapes diferentes em {q}: {arr_2000.shape} vs {arr_2022.shape}")

    diff = arr_2022 - arr_2000

    results[q] = {
        "diff": diff,
        "extent": extent_2000,
        "mean_diff": np.nanmean(diff),
        "median_diff": np.nanmedian(diff),
        "min_diff": np.nanmin(diff),
        "max_diff": np.nanmax(diff),
    }

for q in quarters:
    r = results[q]
    print(f"\n{q} | {band_name} | 2022 - 2000")
    print("mean   :", r["mean_diff"])
    print("median :", r["median_diff"])
    print("min    :", r["min_diff"])
    print("max    :", r["max_diff"])

all_diff = np.concatenate([
    results[q]["diff"][~np.isnan(results[q]["diff"])].ravel()
    for q in quarters
])

common_vmax = np.nanpercentile(np.abs(all_diff), 98)

fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)

for ax, q in zip(axes, quarters):
    im = plot_diff_map(
        ax=ax,
        diff=results[q]["diff"],
        extent=results[q]["extent"],
        title=f"{band_name}: 2022 - 2000 ({q})",
        cmap="RdBu_r",
        vmax=common_vmax,
    )
    jundiai.to_crs("EPSG:4326").boundary.plot(ax=ax, color="black", linewidth=1)

cbar = fig.colorbar(im, ax=axes, shrink=0.9)
cbar.set_label(f"Δ {band_name} (2022 - 2000)")
plt.show()

In [ ]:
raster_q1_2000 = base / "jundiai_2000Q1_ndvi_temp.tif"
raster_q1_2022 = base / "jundiai_2022Q1_ndvi_temp.tif"

raster_q2_2000 = base / "jundiai_2000Q2_ndvi_temp.tif"
raster_q2_2022 = base / "jundiai_2022Q2_ndvi_temp.tif"

In [ ]:
# =====================================
# 2) FUNÇÕES
# =====================================
def read_band_by_description(src, band_name):
    desc = list(src.descriptions)
    if band_name not in desc:
        raise ValueError(f"Banda '{band_name}' não encontrada em {src.name}. Bandas: {desc}")
    return desc.index(band_name) + 1

def clip_band_to_polygon(raster_path, polygon_gdf, band_name="tempC"):
    with rasterio.open(raster_path) as src:
        band_idx = read_band_by_description(src, band_name)

        poly = polygon_gdf.to_crs(src.crs)
        geoms = [geom.__geo_interface__ for geom in poly.geometry]

        clipped, transform = mask(src, geoms, crop=True, filled=True, indexes=band_idx)

        arr = clipped.astype("float32")
        if arr.ndim == 3:
            arr = arr[0]

        nodata = src.nodata
        if nodata is not None:
            arr[arr == nodata] = np.nan

        h, w = arr.shape
        left = transform.c
        top = transform.f
        right = left + transform.a * w
        bottom = top + transform.e * h
        extent = [left, right, bottom, top]

        return arr, extent, src.crs

def plot_one_map(diff, extent, title, polygon_gdf=None, crs="EPSG:4326"):
    vmax = np.nanpercentile(np.abs(diff), 98)

    fig, ax = plt.subplots(figsize=(8, 8))
    im = ax.imshow(
        np.ma.masked_invalid(diff),
        extent=extent,
        origin="upper",
        cmap="RdBu_r",
        vmin=-vmax,
        vmax=vmax
    )

    if polygon_gdf is not None:
        polygon_gdf.to_crs(crs).boundary.plot(ax=ax, color="black", linewidth=1)

    ax.set_title(title)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    plt.colorbar(im, ax=ax, label="Δ tempC (2022 - 2000)")
    plt.show()

# =====================================
# 3) POLÍGONO DE JUNDIAÍ
# =====================================
jundiai = gpd.read_file(municipios_path)
jundiai = jundiai[jundiai["CD_MUN"].astype(str) == "3525904"].copy()

if jundiai.empty:
    raise ValueError("Não encontrei Jundiaí com CD_MUN = 3525904.")

# =====================================
# 4) ESCOLHER A BANDA
# =====================================
band_name = "tempC"   # troque para "ndvi" se quiser

# =====================================
# 5) Q1
# =====================================
arr_q1_2000, extent_q1, crs_q1 = clip_band_to_polygon(raster_q1_2000, jundiai, band_name=band_name)
arr_q1_2022, _, _ = clip_band_to_polygon(raster_q1_2022, jundiai, band_name=band_name)

if arr_q1_2000.shape != arr_q1_2022.shape:
    raise ValueError(f"Shapes diferentes no Q1: {arr_q1_2000.shape} vs {arr_q1_2022.shape}")

diff_q1 = arr_q1_2022 - arr_q1_2000

print("Q1")
print("mean   :", np.nanmean(diff_q1))
print("median :", np.nanmedian(diff_q1))
print("min    :", np.nanmin(diff_q1))
print("max    :", np.nanmax(diff_q1))

plot_one_map(
    diff=diff_q1,
    extent=extent_q1,
    title=f"{band_name}: 2022 - 2000 (Q1)",
    polygon_gdf=jundiai,
    crs=crs_q1
)

# =====================================
# 6) Q2
# =====================================
arr_q2_2000, extent_q2, crs_q2 = clip_band_to_polygon(raster_q2_2000, jundiai, band_name=band_name)
arr_q2_2022, _, _ = clip_band_to_polygon(raster_q2_2022, jundiai, band_name=band_name)

if arr_q2_2000.shape != arr_q2_2022.shape:
    raise ValueError(f"Shapes diferentes no Q2: {arr_q2_2000.shape} vs {arr_q2_2022.shape}")

diff_q2 = arr_q2_2022 - arr_q2_2000

print("\nQ2")
print("mean   :", np.nanmean(diff_q2))
print("median :", np.nanmedian(diff_q2))
print("min    :", np.nanmin(diff_q2))
print("max    :", np.nanmax(diff_q2))

plot_one_map(
    diff=diff_q2,
    extent=extent_q2,
    title=f"{band_name}: 2022 - 2000 (Q2)",
    polygon_gdf=jundiai,
    crs=crs_q2
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from pathlib import Path

# =====================================
# 1) CAMINHOS
# =====================================
#base = Path(r"D:\Users\ivan.cavalcanti\Documents\Projects\extract_landsat\outputs")

rasters = {
    "Q1": {
        "2000": base / "jundiai_2000Q1_ndvi_temp.tif",
        "2022": base / "jundiai_2022Q1_ndvi_temp.tif",
    },
    "Q2": {
        "2000": base / "jundiai_2000Q2_ndvi_temp.tif",
        "2022": base / "jundiai_2022Q2_ndvi_temp.tif",
    },
    "Q3": {
        "2000": base / "jundiai_2000Q3_ndvi_temp.tif",
        "2022": base / "jundiai_2022Q3_ndvi_temp.tif",
    },
    "Q4": {
        "2000": base / "jundiai_2000Q4_ndvi_temp.tif",
        "2022": base / "jundiai_2022Q4_ndvi_temp.tif",
    },
}

#municipios_path = Path(r"D:\Users\ivan.cavalcanti\Documents\Projects\extract_landsat\shapes\municipios_sp.shp")

# =====================================
# 2) FUNÇÕES
# =====================================
def read_band_by_description(src, band_name):
    desc = list(src.descriptions)
    if band_name not in desc:
        raise ValueError(f"Banda '{band_name}' não encontrada em {src.name}. Bandas: {desc}")
    return desc.index(band_name) + 1

def clip_band_to_polygon(raster_path, polygon_gdf, band_name="tempC"):
    with rasterio.open(raster_path) as src:
        band_idx = read_band_by_description(src, band_name)

        poly = polygon_gdf.to_crs(src.crs)
        geoms = [geom.__geo_interface__ for geom in poly.geometry]

        clipped, transform = mask(src, geoms, crop=True, filled=True, indexes=band_idx)

        arr = clipped.astype("float32")
        if arr.ndim == 3:
            arr = arr[0]

        nodata = src.nodata
        if nodata is not None:
            arr[arr == nodata] = np.nan

        h, w = arr.shape
        left = transform.c
        top = transform.f
        right = left + transform.a * w
        bottom = top + transform.e * h
        extent = [left, right, bottom, top]

        return arr, extent, src.crs

# =====================================
# 3) POLÍGONO DE JUNDIAÍ
# =====================================
jundiai = gpd.read_file(municipios_path)
jundiai = jundiai[jundiai["CD_MUN"].astype(str) == "3525904"].copy()

if jundiai.empty:
    raise ValueError("Não encontrei Jundiaí com CD_MUN = 3525904.")

# =====================================
# 4) ESCOLHER BANDA
# =====================================
band_name = "tempC"   # troque para "ndvi" se quiser
colorbar_label = "Δ tempC (2022 - 2000)" if band_name == "tempC" else "Δ NDVI (2022 - 2000)"

# =====================================
# 5) CALCULAR TODOS OS DIFERENCIAIS
# =====================================
results = {}

for quarter in ["Q1", "Q2", "Q3", "Q4"]:
    arr_2000, extent, crs = clip_band_to_polygon(rasters[quarter]["2000"], jundiai, band_name=band_name)
    arr_2022, _, _ = clip_band_to_polygon(rasters[quarter]["2022"], jundiai, band_name=band_name)

    if arr_2000.shape != arr_2022.shape:
        raise ValueError(f"Shapes diferentes no {quarter}: {arr_2000.shape} vs {arr_2022.shape}")

    diff = arr_2022 - arr_2000

    results[quarter] = {
        "diff": diff,
        "extent": extent,
        "crs": crs
    }

# =====================================
# 6) ESCALA COMUM PARA OS 4 MAPAS
# =====================================
all_diff = np.concatenate([
    results[q]["diff"][~np.isnan(results[q]["diff"])].ravel()
    for q in ["Q1", "Q2", "Q3", "Q4"]
])

common_vmax = np.nanpercentile(np.abs(all_diff), 98)

# =====================================
# 7) PLOTAR 4 MAPAS SEPARADOS
# =====================================
for quarter in ["Q1", "Q2", "Q3", "Q4"]:
    diff = results[quarter]["diff"]
    extent = results[quarter]["extent"]
    crs = results[quarter]["crs"]

    print(f"\n{quarter}")
    print("mean   :", np.nanmean(diff))
    print("median :", np.nanmedian(diff))
    print("min    :", np.nanmin(diff))
    print("max    :", np.nanmax(diff))

    fig, ax = plt.subplots(figsize=(8, 8))
    im = ax.imshow(
        np.ma.masked_invalid(diff),
        extent=extent,
        origin="upper",
        cmap="RdBu_r",
        vmin=-common_vmax,
        vmax=common_vmax
    )

    jundiai.to_crs(crs).boundary.plot(ax=ax, color="black", linewidth=1)

    ax.set_title(f"{band_name}: 2022 - 2000 ({quarter})")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    plt.colorbar(im, ax=ax, label=colorbar_label)
    plt.show()